# 08a Clinician model-choice CAM PDFs: CLS vs GAP

This notebook creates two dermatologist-facing PDFs:

1. **Typical melanoma cases** using block `-1`
2. **Typical nevus cases** using block `-10`

Each page contains one image with two rows:
- **CLS model**
- **GAP model**

Each row contains:
- RGB image with lesion outline
- one Grad-CAM map labeled as XAI

Purpose: decide whether **CLS** or **GAP** gives more clinically meaningful explanations before comparing CAM variants or difficult cases.


In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
import numpy as np
from PIL import Image, ImageDraw, ImageFont

print("Python:", sys.executable)


## 1. Configuration

In [ ]:
def find_repo_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path("/storage/homefs/cn21m021/projects/master-thesis"),
        Path("/Users/choekyelnyungmartsang/Developer/master-thesis"),
    ]
    for p in candidates:
        if (p / "scripts" / "generate_finer_cam_panderm.py").exists():
            return p.resolve()
    raise FileNotFoundError("Could not find repo root. Set REPO_ROOT manually.")

REPO_ROOT = find_repo_root()
print("REPO_ROOT:", REPO_ROOT)

HAM_ROOT = REPO_ROOT / "data" / "HAM10000"
MEL_NV_ROOT = HAM_ROOT / "mel_nv"

CLEAN_CSV = MEL_NV_ROOT / "ham_mel_nv_clean.csv"
CANDIDATE_CSV = (
    REPO_ROOT / "outputs" / "mel_nv" / "feature_space_difficult_cases_last_block"
    / "clinician_curation_candidates" / "clinician_curation_candidates_combined_cls_gap.csv"
)

OUT_ROOT = REPO_ROOT / "outputs" / "mel_nv" / "clinician_model_choice_08a"
CSV_OUT_DIR = OUT_ROOT / "csv"
PANEL_OUT_DIR = OUT_ROOT / "panels"
PDF_OUT_DIR = OUT_ROOT / "pdf"
for d in [CSV_OUT_DIR, PANEL_OUT_DIR, PDF_OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLS_CKPT = REPO_ROOT / "external" / "checkpoints4" / "checkpoint-best-cls.pth"
GAP_CKPT = REPO_ROOT / "external" / "checkpoints4" / "checkpoint-best-gap.pth"

SCENARIOS = {
    "CLS": {"checkpoint": CLS_CKPT, "pooling": "cls", "display": "CLS model"},
    "GAP": {"checkpoint": GAP_CKPT, "pooling": "mean", "display": "GAP model"},
}

N_MEL = 10
N_NV = 10
DRY_RUN = False

CAM_METHOD = "gradcam"
MEL_BLOCK = -1
NV_BLOCK = -10

CLASS_NAMES = "MEL,NV"
A_CLASS = "MEL"
B_CLASS = "NV"
PANEL_ITEMS = "rgb_gt_mask,gradcam_a"
PANEL_SUFFIX = PANEL_ITEMS.replace(",", "_")

print("CLEAN_CSV:", CLEAN_CSV.exists(), CLEAN_CSV)
print("CANDIDATE_CSV:", CANDIDATE_CSV.exists(), CANDIDATE_CSV)
print("CLS_CKPT:", CLS_CKPT.exists(), CLS_CKPT)
print("GAP_CKPT:", GAP_CKPT.exists(), GAP_CKPT)


## 2. Select typical MEL and NV cases

In [ ]:
def standardize_label(x):
    if pd.isna(x):
        return x
    s = str(x).strip().upper()
    if s in {"MEL", "MELANOMA"}:
        return "MEL"
    if s in {"NV", "NEVUS", "NEVI"}:
        return "NV"
    return s

def ensure_required_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "image_id" not in df.columns:
        if "isic_id" in df.columns:
            df["image_id"] = df["isic_id"].astype(str)
        elif "image_rel_path" in df.columns:
            df["image_id"] = df["image_rel_path"].astype(str).map(lambda p: Path(p).stem)
        elif "image" in df.columns:
            df["image_id"] = df["image"].astype(str).map(lambda p: Path(p).stem)
        else:
            raise ValueError("Could not infer image_id.")

    if "image_rel_path" not in df.columns:
        if "image" in df.columns:
            df["image_rel_path"] = df["image"].astype(str)
        else:
            df["image_rel_path"] = df["image_id"].astype(str) + ".jpg"

    if "gt_label" not in df.columns:
        if "dx" in df.columns:
            df["gt_label"] = df["dx"].map(standardize_label)
        elif "label_name" in df.columns:
            df["gt_label"] = df["label_name"].map(standardize_label)
        else:
            raise ValueError("Could not infer gt_label.")

    df["gt_label"] = df["gt_label"].map(standardize_label)

    if "mask_rel_path" not in df.columns:
        df["mask_rel_path"] = df["image_id"].astype(str).map(lambda x: f"masks/{x}_segmentation.png")

    return df

clean_df = ensure_required_columns(pd.read_csv(CLEAN_CSV))
print(clean_df.shape)
display(clean_df.head())
print(clean_df["gt_label"].value_counts(dropna=False))


In [ ]:
def select_from_candidate_csv(candidate_csv: Path, clean_df: pd.DataFrame):
    if not candidate_csv.exists():
        return None

    cand = ensure_required_columns(pd.read_csv(candidate_csv))

    clean_base = clean_df[["image_id", "image_rel_path", "mask_rel_path", "gt_label"]].drop_duplicates("image_id")
    cand = cand.drop(columns=[c for c in ["image_rel_path", "mask_rel_path", "gt_label"] if c in cand.columns], errors="ignore")
    cand = cand.merge(clean_base, on="image_id", how="left")

    if "selection_group" not in cand.columns:
        return None

    mel = cand[(cand["selection_group"] == "nearest_mel_centroid") & (cand["gt_label"] == "MEL")].drop_duplicates("image_id").head(N_MEL).copy()
    nv = cand[(cand["selection_group"] == "nearest_nv_centroid") & (cand["gt_label"] == "NV")].drop_duplicates("image_id").head(N_NV).copy()

    if len(mel) == 0 or len(nv) == 0:
        return None

    mel["selection_reason"] = "nearest_mel_centroid"
    nv["selection_reason"] = "nearest_nv_centroid"
    return mel, nv

selected = select_from_candidate_csv(CANDIDATE_CSV, clean_df)
if selected is None:
    mel_df = clean_df[clean_df["gt_label"] == "MEL"].drop_duplicates("image_id").head(N_MEL).copy()
    nv_df = clean_df[clean_df["gt_label"] == "NV"].drop_duplicates("image_id").head(N_NV).copy()
    mel_df["selection_reason"] = "fallback_first_mel"
    nv_df["selection_reason"] = "fallback_first_nv"
else:
    mel_df, nv_df = selected

keep_cols = ["image_id", "image_rel_path", "mask_rel_path", "gt_label", "selection_reason"]
mel_df = mel_df[keep_cols].reset_index(drop=True)
nv_df = nv_df[keep_cols].reset_index(drop=True)

mel_csv = CSV_OUT_DIR / "08a_typical_mel_model_choice.csv"
nv_csv = CSV_OUT_DIR / "08a_typical_nv_model_choice.csv"
mel_df.to_csv(mel_csv, index=False)
nv_df.to_csv(nv_csv, index=False)

print("MEL:", len(mel_df), mel_csv)
display(mel_df)
print("NV:", len(nv_df), nv_csv)
display(nv_df)


## 3. Generate CAM panels with existing script

In [ ]:
def run_command(cmd, dry_run=False):
    print("\n$", " ".join(str(x) for x in cmd))
    if dry_run:
        return
    result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}")

def generate_panels_for_subset(subset_name: str, csv_path: Path, target_block_index: int):
    out_dirs = {}
    n = len(pd.read_csv(csv_path))

    for model_key, cfg in SCENARIOS.items():
        out_dir = PANEL_OUT_DIR / subset_name / model_key.lower() / f"block_{target_block_index}"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_dirs[model_key] = out_dir

        cmd = [
            sys.executable, "-m", "scripts.generate_finer_cam_panderm",
            "--csv", str(csv_path),
            "--image_col", "image_rel_path",
            "--img_dir", str(HAM_ROOT),
            "--gt_col", "gt_label",
            "--checkpoint", str(cfg["checkpoint"]),
            "--checkpoint_model_type", "panderm",
            "--class_names", CLASS_NAMES,
            "--pooling", cfg["pooling"],
            "--out_dir", str(out_dir),
            "--num_samples", str(n),
            "--method", CAM_METHOD,
            "--compare_mode", "gt_pair",
            "--A", A_CLASS,
            "--B", B_CLASS,
            "--alpha", "0.8",
            "--panel_items", PANEL_ITEMS,
            "--mask_root", str(HAM_ROOT),
            "--mask_col", "mask_rel_path",
            "--target_block_index", str(target_block_index),
            "--clinician_labels",
            "--model_display_name", cfg["display"],
            "--save_json",
            "--save_raw_cams",
        ]
        run_command(cmd, dry_run=DRY_RUN)

    return out_dirs

mel_panel_dirs = generate_panels_for_subset("mel_block_minus1", mel_csv, MEL_BLOCK)
nv_panel_dirs = generate_panels_for_subset("nv_block_minus10", nv_csv, NV_BLOCK)

mel_panel_dirs, nv_panel_dirs


## 4. Assemble final PDFs

In [ ]:
def load_font(size: int, bold: bool = False):
    candidates = [
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/Library/Fonts/Arial Bold.ttf" if bold else "/Library/Fonts/Arial.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for p in candidates:
        try:
            return ImageFont.truetype(p, size=size)
        except Exception:
            pass
    return ImageFont.load_default()

def find_panel_png(panel_dir: Path, image_id: str) -> Path:
    candidates = sorted(panel_dir.glob(f"{image_id}*_{PANEL_SUFFIX}.png"))
    if len(candidates) == 0:
        candidates = sorted(panel_dir.glob(f"*{image_id}*{PANEL_SUFFIX}.png"))
    if len(candidates) == 0:
        raise FileNotFoundError(f"No panel PNG for {image_id} in {panel_dir}")
    return candidates[0]

def add_row_label(panel_img: Image.Image, label: str) -> Image.Image:
    label_w = 180
    canvas = Image.new("RGB", (panel_img.width + label_w, panel_img.height), "white")
    draw = ImageDraw.Draw(canvas)
    font = load_font(28, bold=True)
    canvas.paste(panel_img, (label_w, 0))
    bbox = draw.textbbox((0, 0), label, font=font)
    tw = bbox[2] - bbox[0]
    th = bbox[3] - bbox[1]
    draw.text(((label_w - tw) / 2, panel_img.height / 2 - th / 2), label, font=font, fill=(0, 0, 0))
    return canvas

def make_page(image_id, gt_label, selection_reason, cls_png, gap_png, block_label, pdf_title):
    cls_panel = Image.open(cls_png).convert("RGB")
    gap_panel = Image.open(gap_png).convert("RGB")

    max_w = max(cls_panel.width, gap_panel.width)
    def pad_width(img, width):
        if img.width == width:
            return img
        canvas = Image.new("RGB", (width, img.height), "white")
        canvas.paste(img, ((width - img.width) // 2, 0))
        return canvas

    cls_row = add_row_label(pad_width(cls_panel, max_w), "CLS")
    gap_row = add_row_label(pad_width(gap_panel, max_w), "GAP")

    margin = 40
    title_h = 120
    gap_y = 28

    page_w = max(cls_row.width, gap_row.width) + 2 * margin
    page_h = title_h + cls_row.height + gap_y + gap_row.height + 2 * margin

    page = Image.new("RGB", (page_w, page_h), "white")
    draw = ImageDraw.Draw(page)

    title_font = load_font(34, bold=True)
    sub_font = load_font(22, bold=False)

    draw.text((margin, margin - 5), pdf_title, font=title_font, fill=(0, 0, 0))
    draw.text(
        (margin, margin + 45),
        f"{image_id} | Ground truth: {gt_label} | {block_label} | Selection: {selection_reason}",
        font=sub_font,
        fill=(70, 70, 70),
    )
    draw.text(
        (margin, margin + 78),
        "Please circle: CLS / GAP / both / neither. Optional: add short reason.",
        font=sub_font,
        fill=(70, 70, 70),
    )

    y = margin + title_h
    page.paste(cls_row, (margin, y))
    y += cls_row.height + gap_y
    page.paste(gap_row, (margin, y))
    return page

def assemble_pdf(subset_df, panel_dirs, block_label, pdf_title, out_pdf):
    pages = []
    for _, row in subset_df.iterrows():
        image_id = str(row["image_id"])
        cls_png = find_panel_png(panel_dirs["CLS"], image_id)
        gap_png = find_panel_png(panel_dirs["GAP"], image_id)
        pages.append(make_page(
            image_id=image_id,
            gt_label=str(row["gt_label"]),
            selection_reason=str(row["selection_reason"]),
            cls_png=cls_png,
            gap_png=gap_png,
            block_label=block_label,
            pdf_title=pdf_title,
        ))

    if not pages:
        raise ValueError("No pages to save.")
    pages[0].save(out_pdf, save_all=True, append_images=pages[1:], resolution=150.0)
    print("Saved:", out_pdf)

mel_pdf = PDF_OUT_DIR / "08a_model_choice_typical_mel_cls_vs_gap_gradcam_block_minus1.pdf"
nv_pdf = PDF_OUT_DIR / "08a_model_choice_typical_nv_cls_vs_gap_gradcam_block_minus10.pdf"

if not DRY_RUN:
    assemble_pdf(
        subset_df=mel_df,
        panel_dirs=mel_panel_dirs,
        block_label="MEL block -1",
        pdf_title="Model-choice review: typical melanoma cases",
        out_pdf=mel_pdf,
    )
    assemble_pdf(
        subset_df=nv_df,
        panel_dirs=nv_panel_dirs,
        block_label="NV block -10",
        pdf_title="Model-choice review: typical nevus cases",
        out_pdf=nv_pdf,
    )
else:
    print("DRY_RUN=True: skipping PDF assembly.")

mel_pdf, nv_pdf


## 5. Annotation sheet

In [ ]:
annotation_rows = []
for subset_name, subset_df in [("typical_mel", mel_df), ("typical_nv", nv_df)]:
    for _, row in subset_df.iterrows():
        annotation_rows.append({
            "subset": subset_name,
            "image_id": row["image_id"],
            "gt_label": row["gt_label"],
            "selection_reason": row["selection_reason"],
            "dermatologist_choice": "",  # CLS / GAP / both / neither
            "comment": "",
        })

annotation_df = pd.DataFrame(annotation_rows)
annotation_csv = CSV_OUT_DIR / "08a_dermatologist_model_choice_annotation_sheet.csv"
annotation_df.to_csv(annotation_csv, index=False)
print("Saved:", annotation_csv)
display(annotation_df.head())
